In [ ]:
# README
# script to analyze intermediate values:
# silent_error() -> function to calculate number of silent errors and intermediate errors
# process_bit_flips() -> calculate the Hamming distance and positions of each bit flip
# intermediate_analyse() -> plots Hamming Distance and Hamming Sum for each Byte of a layer, for each number (0-9) for the target execution time
# df1, df2 -> the 2 input Dataframes, generated from the no_glitch and faulty data


import pandas as pd
import ast 
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import os
import sys
sys.path.insert(1, 'ml-fault-attacks-on-tinyml/notebooks')

columns= ["CNN_out", "Maxpool_out", "FC_out"]


def calculate_bit_flips(num1, num2):
    """Calculate bit flips between two integers."""
    xor_result= num1 ^ num2
    bit_flips_count = bin(xor_result).count('1')        # total count of bits flipped 
    flipped_positons= [index for index, bit in enumerate(bin(xor_result)[:1:-1]) if bit== '1']  # list of positions where bit has been flipped
    return bit_flips_count, flipped_positons



def silent_error(result_df):
    sil_err_count= 0                # count of silent errors-> bits have flipped in intermediate layers but final result is correct
    intermediate_error= 0           # count of total intermediate errors-> bits flipped regardless of final result

    for index in range(len(result_df)):
            pred= result_df.loc[index, "Predicted"]
            if isinstance(pred, str):
                try:
                    pred = ast.literal_eval(pred)  # Convert string to list
                except (ValueError, SyntaxError):
                    pred = []
            if pred:
                pred = pred[0]  # Extract the first element if pred is a list with values
            else:
                pred = None 
            
            # print(type(pred))
            if result_df.loc[index, "Fail"] == 0:
                if ((result_df.loc[index, "CNN_out_Hamming_sum"] != 0 ) or (result_df.loc[index, "Maxpool_out_Hamming_sum"] != 0 ) or (result_df.loc[index, "FC_out_Hamming_sum"] != 0 )) and (result_df.loc[index, "Actual"] == pred):
                    sil_err_count+=1

                if ((result_df.loc[index, "CNN_out_Hamming_sum"] != 0 ) or (result_df.loc[index, "Maxpool_out_Hamming_sum"] != 0 ) or (result_df.loc[index, "FC_out_Hamming_sum"] != 0 )):
                    intermediate_error+=1
                
    print("intermediate_error for {}: {}".format(per, intermediate_error))

def process_bit_flips():
    """calculate no. of bit flips"""
        # check if both dataframes have required columns

        #df1-> no glitch 
        #df2-> with fault

    for col in columns:
        if col not in df1.columns or col not in df2.columns:
            raise ValueError("Column '{}' not found in one or both files.".format(col))
        
        # Initialize DataFrame to store results
    result_df = pd.DataFrame()
    
    for col in columns:
        # Process row-by-row for the given column
        bit_flips_list = []
        positions_list= []
        
        for index in range(len(df1)):
            list1 = df1.loc[index, col]
            list2 = df2.loc[index, col]
            
            # Convert strings to lists if needed
            if isinstance(list1, str):
                list1 = eval(list1)
            if isinstance(list2, str):
                list2 = eval(list2)
        # MSB flips
            position_dict={}        # to store index of no. that has been flipped and the position of bit flips in that number
            row_flips=[]
            for index2, (num1, num2) in enumerate(zip(list1, list2)):
                    bit_flips, flipped_positions = calculate_bit_flips(num1, num2)
                    row_flips.append(bit_flips)   # appending total bits flipped for each number in the list for 1 row
                    if num1!=num2:
                        position_dict[index2]= flipped_positions
        
            positions_list.append(position_dict)
            bit_flips_list.append(row_flips)

        # Add the result to the new DataFrame
        result_df["{}_Hamming Distance".format(col)] = bit_flips_list
        result_df["{}_Position_Flipped_Bit".format(col)]= positions_list
        result_df["{}_Hamming_sum".format(col)]= [sum(sublist) for sublist in bit_flips_list]
    
    result_df["CNN_no_glitch"]= df1["CNN_out"]
    result_df["CNN_faulty"]= df2["CNN_out"]

    result_df["Maxpool_no_glitch"]= df1["Maxpool_out"]
    result_df["Maxpool_faulty"]= df2["Maxpool_out"]

    result_df["FC_no_glitch"]= df1["FC_out"]
    result_df["FC_faulty"]= df2["FC_out"]  

    result_df["Predicted"]= df2["Prediction"]
    result_df["Actual"]= df2["Actual"]
    result_df["Fail"]= df2["Fail"]
    
    file_name = "confusion_matrices/tiny_cnn/data/intermediate_values/data/inter_{}%.csv".format(per)
    result_df.to_csv(file_name, index=False)
    # result_df.to_csv("C:\Users\Wenjie Xiong\Documents\\fault-injection-setup\src\Fault_Injection_ML_Algorithms\ML_data\\fine_grain_search\img_classification\intermediate_value\inter_{}%.csv".format(per), index= False)

    # intermediate_analyse(result_df)
    # silent_error(result_df)
    return result_df

# boxplot for each number
def intermediate_analyse(result_df):

    nums= [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    # nums= [6, 7, 8, 9]      # the numbers (predicted) to plot for analysis
    # print(result_df.loc[2, "Predicted"][1], type(result_df.loc[2, "Predicted"]))    

    for num in nums:
        count=0
        analysedf_CNN= pd.DataFrame()       # for each number, this df stores the hamming distance values for CNN layer and same follows for other 2 layers. 
        analysedf_Max= pd.DataFrame()
        analysedf_FC= pd.DataFrame()

        for j in range(len(result_df)):
            pred_list= result_df.loc[j, "Predicted"]
            pred_value = ast.literal_eval(pred_list)  # Converts "[8]" into [8]
            # print(pred_value, type(pred_value)) 
            if pred_value:
                if pred_value[0] == num:
                        
                        count+=1
                        analysedf_CNN = pd.concat([analysedf_CNN, pd.DataFrame({"Num {}: CNN_out_Hamming Distance".format(num): [result_df.loc[j, "CNN_out_Hamming Distance"]]}, index=[j])])
                        analysedf_Max = pd.concat([analysedf_Max, pd.DataFrame({"Num {}: Maxpool_out_Hamming Distance".format(num): [result_df.loc[j, "Maxpool_out_Hamming Distance"]]}, index=[j])])
                        analysedf_FC = pd.concat([analysedf_FC, pd.DataFrame({"Num {}: FC_out_Hamming Distance".format(num): [result_df.loc[j, "FC_out_Hamming Distance"]]}, index=[j])])
        print(count)
        df_exploded_CNN = pd.DataFrame(analysedf_CNN["Num {}: CNN_out_Hamming Distance".format(num)].tolist())  # converts all hamming distance values to lists, so plotting becomes easy
        df_exploded_Max = pd.DataFrame(analysedf_Max["Num {}: Maxpool_out_Hamming Distance".format(num)].tolist())
        df_exploded_FC = pd.DataFrame(analysedf_FC["Num {}: FC_out_Hamming Distance".format(num)].tolist())

        # boxplot(num, analysedf_CNN, df_exploded_CNN, df_exploded_Max, df_exploded_FC)


def boxplot(num, analysedf_CNN, df_exploded_CNN, df_exploded_Max, df_exploded_FC): # plotting boxplots for the 3 layers

    num_cols = len(analysedf_CNN.columns)
    colors = [cm.viridis(i / num_cols) for i in range(num_cols)]

    sums_CNN = df_exploded_CNN.sum(axis=1)      #calculate hamming sum for all layers
    sums_Max = df_exploded_Max.sum(axis=1)
    sums_FC = df_exploded_FC.sum(axis=1)
#boxplot for CNN layer
    plt.figure(figsize=(10, 6))
    box= df_exploded_CNN.boxplot(patch_artist=True)
    # colors = ['lightblue', 'lightgreen', 'lightcoral']
    for patch, color in zip(box.artists, colors):  
        patch.set_facecolor(color) 

    plt.title("CNN Plot for Num {} Showing Min, Max, and Median for each Byte".format(num))
    plt.ylabel("Hamming Distance")
    plt.xticks(rotation=45)  # Rotate column names for readability
    plt.legend()
    plt.show()
    # plt.savefig("src\Fault_Injection_ML_Algorithms\ML_data\\fine_grain_search\img_classification\plots\intermediate_analysis\\ham_analysis_{}_CNN.pdf".format(per, num), format="pdf", bbox_inches="tight")

    #plot for sum
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(sums_CNN) + 1), sums_CNN, marker='o', color='blue', linestyle='dashed', label="Sum")
    plt.xlabel("Data Index")
    plt.ylabel("Sum of Hamming Distances")
    plt.title("Sum of CNN Output per Data Point")
    plt.legend()
    plt.grid(True)
    plt.show()
    # plt.savefig("src\Fault_Injection_ML_Algorithms\ML_data\\fine_grain_search\img_classification\plots\intermediate_analysis\ham_analysis_{}_CNN_sum.pdf".format(num), format="pdf", bbox_inches="tight")

#boxplot for Maxpool layer

    plt.figure(figsize=(10, 6))
    box2= df_exploded_Max.boxplot(patch_artist=True)
    # colors = ['lightblue', 'lightgreen', 'lightcoral']
    for patch, color in zip(box2.artists, colors):  
        patch.set_facecolor(color) 
    plt.title("Maxpool Plot for Num {} Showing Min, Max, and Median for each Byte".format(num))
    plt.ylabel("Hamming Distance")
    plt.xticks(rotation=45)  
    plt.show()
    # plt.savefig("src\Fault_Injection_ML_Algorithms\ML_data\\fine_grain_search\img_classification\plots\intermediate_analysis\ham_analysis_{}_Maxpool.pdf".format(num), format="pdf", bbox_inches="tight")
    
    #plot for sum
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(sums_Max) + 1), sums_Max, marker='o', color='blue', linestyle='dashed', label="Sum")
    plt.xlabel("Data Index")
    plt.ylabel("Sum of Hamming Distances")
    plt.title("Sum of Maxpool Output per Data Point")
    plt.legend()
    plt.grid(True)
    p,t.show()
    # plt.savefig("src\Fault_Injection_ML_Algorithms\ML_data\\fine_grain_search\img_classification\plots\intermediate_analysis\ham_analysis_{}_Maxpool_sum.pdf".format(num), format="pdf", bbox_inches="tight")

#boxplot for FC layer

    plt.figure(figsize=(10, 6))
    box3= df_exploded_FC.boxplot(patch_artist=True)
    # colors = ['lightblue', 'lightgreen', 'lightcoral']
    for patch, color in zip(box3.artists, colors):  
        patch.set_facecolor(color) 
    plt.title("FC Plot for Num {} Showing Min, Max, and Median for each Byte".format(num))
    plt.ylabel("Hamming Distance")
    plt.xticks(rotation=45)  
    plt.show()
    # plt.savefig("src\Fault_Injection_ML_Algorithms\ML_data\\fine_grain_search\img_classification\plots\intermediate_analysis\ham_analysis_{}_FC.pdf".format(num), format="pdf", bbox_inches="tight")

    #plot for sum
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(sums_FC) + 1), sums_FC, marker='o', color='blue', linestyle='dashed', label="Sum")
    plt.xlabel("Data Index")
    plt.ylabel("Sum of Hamming Distances")
    plt.title("Sum of FC Output per Data Point")
    plt.legend()
    plt.grid(True)
    plt.show()
    

percentage= [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
# percentage = [0.01*x for x in range(0, 105, 5)]
filename = "confusion_matrices/tiny_cnn/data/pinata/img_classification_layer1_trainset"
for per in percentage:
    df1 = pd.read_csv(filename + f"_no_glitch2.csv") # one file with no glitch
    df2 = pd.read_csv(filename + "_{}%_run3_95e-9.csv".format(per)) 
    # df1 = pd.read_csv( 'confusion_matrices/tiny_cnn/data/image_cnn_run_unprotected_no_fault.csv') # one file with no glitch
    # df2 = pd.read_csv(filename + f"_{per:.4f}_start_time_run_office.csv") # another file with fault injection
    result = process_bit_flips()
    # silent_error(result)


print("results saved to csv!")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import os
import math
from tqdm.notebook import tqdm

def popcount8(x: int) -> int:
    return bin(x).count("1")      
        
        
def plot_heatmap_hweight(csv_file, save_path, inj_number):
    # --- load & preprocess ---
    df = pd.read_csv(csv_file)
    df = df[df['CNN_out_Hamming Distance'] != '[]'].copy()
    df['CNN_out_Hamming Distance'] = df['CNN_out_Hamming Distance'].apply(ast.literal_eval)
    df['CNN_faulty'] = df['CNN_faulty'].apply(ast.literal_eval)

    # expand distance into columns
    max_len = df['CNN_out_Hamming Distance'].map(len).max()
    hamming_cols = []
    for i in range(max_len):
        c = f'hamming_{i}'
        df[c] = df['CNN_out_Hamming Distance'].map(lambda L: L[i] if i < len(L) else np.nan)
        hamming_cols.append(c)
    df[hamming_cols] = df[hamming_cols].apply(pd.to_numeric, errors='coerce')

    # --- create compact 2×5 grid ---
    actuals = sorted(df['Actual'].unique())
    rows, cols = 2, 5
    fig, axes = plt.subplots(rows, cols,
                             figsize=(cols * 4.0, rows * 3.0),
                             squeeze=False)
    fig.suptitle(f'Hamming Distance Heatmaps for {inj_number}% Injection Time', fontsize=20)

    im = None
    for idx, actual in enumerate(actuals):
        r, c = divmod(idx, cols)
        ax = axes[r][c]
        sub = df[df['Actual'] == actual].sort_values('Predicted')
        mat = sub[hamming_cols].values.astype(float)

        # compute true Hamming-weight sum & average
        faulty = sub['CNN_faulty'].tolist()
        weights_sum = [sum(popcount8(x & 0xFF) for x in row) for row in faulty]
        # weights_sum = [sum((x & 0xFF).bit_count() for x in row) for row in faulty]
        weights_avg = [w / len(row) for w, row in zip(weights_sum, faulty)]

        # draw heatmap
        im = ax.imshow(mat, cmap='Reds', aspect='auto', vmin=0, vmax=8)
        ax.set_title(f'Actual = {actual}', pad=8, fontsize=14)
        ax.set_xlabel('Position', fontsize=12)
        ax.set_ylabel('Predicted', fontsize=12)

        # group separators
        preds = sub['Predicted'].astype(str).tolist()
        breaks = [0] + [i+1 for i in range(len(preds)-1) if preds[i] != preds[i+1]]
        labels = [preds[i] for i in breaks]
        ax.set_yticks(breaks)
        ax.set_yticklabels(labels, fontsize='medium')
        for b in breaks[1:]:
            ax.axhline(b - 0.5, color='white', linestyle='--', linewidth=0.7)

        # X-ticks
        nc = mat.shape[1]
        nt = min(8, nc)
        xt = np.linspace(0, nc-1, nt).astype(int)
        ax.set_xticks(xt)
        ax.set_xticklabels(xt, fontsize='medium')

        # twin-y for (avg, sum)
        ax2 = ax.twinx()
        ax2.set_ylim(ax.get_ylim())
        rows_idx = np.arange(mat.shape[0])
        ax2.set_yticks(rows_idx)
        # format as (avg, sum)
        tup_labels = [f"({avg:.2f},{s})" for avg, s in zip(weights_avg, weights_sum)]
        ax2.set_yticklabels(tup_labels, fontsize='x-small')
        ax2.set_ylabel('(avg, sum)', rotation=270, labelpad=12)

    # disable unused axes
    for i in range(len(actuals), rows*cols):
        axes.flatten()[i].axis('off')

    # tighten layout + colorbar
    plt.subplots_adjust(left=0.035, right=0.844, top=0.892, bottom=0.11,
                        wspace=0.723, hspace=0.449)
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    fig.colorbar(im, cax=cax, label='Hamming Distance')

    # plt.savefig('{}\heatmap_h_distance_weight_{}.pdf'.format(save_path, inj_number))
    # plt.show()

def plot_heatmap(csv_file, save_path, inj_number):
    df = pd.read_csv(csv_file)

    # Filter out rows where 'CNN_out_Hamming Distance' is '[]'
    df = df[df['CNN_out_Hamming Distance'] != '[]'].copy()

    # Convert string representation of list to actual list
    df.loc[:, 'CNN_out_Hamming Distance'] = df['CNN_out_Hamming Distance'].apply(ast.literal_eval)

    # Expand the 'CNN_out_Hamming Distance' list into separate columns
    max_len = df['CNN_out_Hamming Distance'].apply(len).max()
    hamming_cols = []
    for i in range(max_len):
        df['hamming_{}'.format(i)] = df['CNN_out_Hamming Distance'].apply(lambda x: x[i] if len(x) > i else np.nan)
        hamming_cols.append('hamming_{i}'.format(i))

    # Convert hamming columns to numeric
    df[hamming_cols] = df[hamming_cols].apply(pd.to_numeric, errors='coerce')

    # Create a new column combining predicted and actual values
    df['Prediction_vs_Actual'] = 'Predicted: ' + df['Predicted'].astype(str) + ', Actual: ' + df['Actual'].astype(str)

    # Group by 'Actual' number and plot heatmap
    for actual_value in df['Actual'].unique():
        subset_df = df[df['Actual'] == actual_value].copy()

        # Sort by predicted value
        subset_df = subset_df.sort_values(by='Predicted')

        # Collect all hamming values for the current actual value
        all_hamming_values = subset_df[hamming_cols].values.astype(float)

        # Get the predicted values for y-axis labels
        predicted_values = subset_df['Predicted'].astype(str).tolist()
        
        # Find the indices where the predicted value changes
        indices = [0] + [i+1 for i in range(len(predicted_values)-1) if predicted_values[i] != predicted_values[i+1]]

        # Get unique predicted values for y-axis labels
        unique_predicted_values = [predicted_values[i] for i in indices]

        # Plot all rows as a single heatmap
        plt.figure(figsize=(15, 10))  # Adjust figure size for better readability
        plt.imshow(all_hamming_values, cmap='ocean', aspect='auto', vmin=0, vmax=8)  # Use 'aspect='auto'' to avoid squishing, set colorbar range
        plt.colorbar(label='Hamming Distance')
        plt.title(f'Hamming Distance for CNN Output (Actual = {actual_value}) - {inj_number}% Injection Time')
        plt.xlabel('CNN Output Position')
        plt.ylabel('Predicted Values')
        
        # Set y-axis ticks and labels
        plt.yticks(indices, unique_predicted_values)
        
        # Add horizontal lines to separate prediction groups
        for index in indices[1:]:
            plt.axhline(y=index-0.5, color='white', linestyle='--')

        # Reduce the number of x-ticks
        num_ticks = 19  # Adjust as needed
        plt.xticks(np.linspace(0, len(hamming_cols) - 1, num_ticks), np.linspace(0, len(hamming_cols) - 1, num_ticks).astype(int))  # Set x-axis ticks to be CNN Output Position
        
        plt.tight_layout()  # Adjust layout to prevent labels from overlapping
        
        # Uncomment this line to display the heatmap instead of saving it
        plt.show() 
        
        # Save the heatmap to a file (comment thsi line to not save)
        plt.savefig('{}\heatmap_actual_{}.pdf'.format(save_path, actual_value))  # Save each heatmap to a PDF file
       


if __name__ == '__main__':
    save_directory = "confusion_matrices/tiny_cnn/data/intermediate_values/plots"
    os.makedirs(save_directory, exist_ok=True)

    # injection_times = [0.01*x for x in range(0, 105, 5)] 
    injection_times = list(range(0, 105, 5))
    # if you want to skip 25%:
    # injection_times.remove(25)

    # injection_times = [0, 10, 50, 60, 65]

    # This will collect a list of (inj_rate, avg_vector)
    avg_bitflips_per_inj = []

    for inj in tqdm(injection_times):
        print(f"Processing Hamming Distance for {inj}% injection time")
        csv_file = f"confusion_matrices/tiny_cnn/data/intermediate_values/data/inter_{inj}%.csv" 
        if not os.path.exists(csv_file):
            print(f"  → {csv_file} not found, skipping.")
            continue

        # 1) your existing per-injection plot:
        plot_heatmap_hweight(csv_file, save_directory, inj)

        # 2) now read & compute per-position average bit-flip:
        df = pd.read_csv(csv_file)
        df = df[df['CNN_out_Hamming Distance'] != '[]'].copy()
        df['CNN_out_Hamming Distance'] = df['CNN_out_Hamming Distance'].apply(ast.literal_eval)

        # expand into columns
        max_len = df['CNN_out_Hamming Distance'].map(len).max()
        cols = [f'hamming_{i}' for i in range(max_len)]
        for i, col in enumerate(cols):
            df[col] = df['CNN_out_Hamming Distance'].map(lambda L: L[i] if i < len(L) else np.nan)
        df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

        # compute the mean over all rows (actual values)
        avg_vec = df[cols].mean(axis=0).values  # shape = (max_len,)
        avg_bitflips_per_inj.append((inj, avg_vec))

    # ----- now plot the “all-injection” average heatmap -----
    if avg_bitflips_per_inj:
        rates, vecs = zip(*avg_bitflips_per_inj)
        matrix = np.vstack(vecs)  # shape = (n_rates, n_positions)

        plt.figure(figsize=(12, 6), dpi=1200)
        im = plt.imshow(matrix, cmap='Reds', aspect='auto', vmin=0, vmax=8)
        cbar = plt.colorbar(im)
        cbar.set_label('Avg Hamming Distance', fontsize=20)
        cbar.ax.tick_params(labelsize=18)

        # y-ticks are the injection rates
        yt = np.arange(len(rates))
        plt.yticks(yt, [f"{r}%" for r in rates], fontsize=18)
        plt.xticks(range(0, 287, 32), fontsize=18)
        plt.xlabel('CNN Output Position', fontsize=20)
        plt.ylabel('Injection Time (%)', fontsize=20)
        plt.title('Average Bit-Flips per Position vs. Injection Time', fontsize=24)
        plt.tight_layout()

        out_file = os.path.join(save_directory, 'avg_bitflips_all_injections_pinata.pdf')
        plt.savefig(out_file)
        # plt.show()
    else:
        print("No valid CSVs found; skipping average‐heatmap.")
